# Week 6 · Task A — Make the files trustworthy
**Integration checkpoint · National Grid Authority case · 5 marks**

This notebook reads the three raw CSVs and the GeoJSON, **audits** them, **fixes** what it finds and writes cleaned copies to `clean/`.
Nothing is edited by hand: every change is a line of code, and every change is logged in a **decisions table** (what · how detected · rule · rows affected).

Run it top to bottom. It also writes `partA_audit.txt` (the printed audit) and `partA_decisions.md` (the decisions table).

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)

def find_root():
    """Folder that holds the three toolkits and data/ - works from week6/assignments or a copy of it."""
    here = Path.cwd().resolve()
    for base in [here, here / "assignments", here.parent, here.parent / "assignments"]:
        if (base / "vizlib.py").exists() and (base / "data").is_dir():
            return base
    raise FileNotFoundError("Open this notebook in the folder that contains vizlib.py, tslib.py, geolib.py and data/")

ROOT = find_root()
sys.path.insert(0, str(ROOT))
DATA, CLEAN = ROOT / "data", ROOT / "clean"
CLEAN.mkdir(exist_ok=True)

import numpy as np, pandas as pd
import vizlib, tslib, geolib
pd.set_option("display.width", 160, "display.max_columns", 30)

AUDIT, DECISIONS = [], []

def say(*parts):
    s = " ".join(str(p) for p in parts)
    print(s); AUDIT.append(s)

def decide(what, how, rule, n, action):
    """One row per decision. `action` is repair / drop / flag / keep."""
    DECISIONS.append({"file": None, "what": what, "how detected": how, "rule applied": rule,
                      "rows affected": n, "action": action})

print("root:", ROOT.name)

root: week6_solution


## 0 · Load the raw files (read-only)

In [2]:
reg_raw   = pd.read_csv(DATA / "district_grid.csv")
daily_raw = pd.read_csv(DATA / "grid_daily.csv")
gen_raw   = pd.read_csv(DATA / "generation_mix.csv")
feats     = geolib.read_geojson(DATA / "regions.geojson")
geo_names = {f["code"]: f["name"] for f in feats}
SENTINELS = [-1, -999, 999, 9999, 99999]      # operator "not available" codes to look for
print(len(reg_raw), "register rows |", len(daily_raw), "daily rows |", len(gen_raw), "generation rows |", len(feats), "GeoJSON features")

25 register rows | 3061 daily rows | 162 generation rows | 24 GeoJSON features


## 1 · `district_grid.csv` — the district register (25 rows, short enough to read)

**Audit first, fix second.** Printed per file: dtypes, key uniqueness, duplicate keys, sentinel values, join misses, and the ranges of anything we will divide by (households, population, area).

In [3]:
say("=" * 72); say("AUDIT 1 - district_grid.csv"); say("=" * 72)
say(f"shape: {reg_raw.shape}")
say("dtypes:"); say(reg_raw.dtypes.to_string())
say("nulls per column:", {c: int(n) for c, n in reg_raw.isna().sum().items() if n})

# key uniqueness / duplicate keys
dup_codes = reg_raw.code[reg_raw.code.duplicated(keep=False)].unique().tolist()
say(f"key `code` unique: {reg_raw.code.is_unique} | duplicated keys: {dup_codes}")
say(reg_raw[reg_raw.code.isin(dup_codes)].to_string())

# a numeric column that did not parse as numeric
el_num = pd.to_numeric(reg_raw.electrification_pct, errors="coerce")
bad_el = reg_raw[el_num.isna() & reg_raw.electrification_pct.notna()]
say(f"electrification_pct dtype = {reg_raw.electrification_pct.dtype}; text entries that are not numbers: "
    f"{dict(zip(bad_el.code, bad_el.electrification_pct))}; genuinely blank: {reg_raw.code[reg_raw.electrification_pct.isna()].tolist()}")

# sentinel scan on every numeric column
num = reg_raw.select_dtypes("number")
hits = {c: reg_raw.code[num[c].isin(SENTINELS)].tolist() for c in num if num[c].isin(SENTINELS).any()}
say("sentinel codes (-1, 999, ...) found in:", hits)

# ranges of everything we will divide by
for c in ["population", "households", "area_km2", "outages"]:
    say(f"range {c:<11}: {reg_raw[c].min():>14,.1f} .. {reg_raw[c].max():>14,.1f}")
pph = reg_raw.population / reg_raw.households
say(f"persons per household: min {pph.min():.2f}  max {pph.max():,.0f}   <- implausible rows: {reg_raw.code[(pph < 3) | (pph > 12)].tolist()}")
say("persons per household of the other rows: "
    f"{pph[(pph >= 3) & (pph <= 12)].min():.2f} .. {pph[(pph >= 3) & (pph <= 12)].max():.2f}")

# join misses
miss_geo = sorted(set(reg_raw.code) - set(geo_names)); miss_reg = sorted(set(geo_names) - set(reg_raw.code))
say(f"join register -> GeoJSON on `code`: register codes with no polygon: {miss_geo}; polygons with no register row: {miss_reg}")
say(f"join register zones vs daily zones: register-only {sorted(set(reg_raw.zone)-set(daily_raw.zone))}, daily-only {sorted(set(daily_raw.zone)-set(reg_raw.zone))}")

AUDIT 1 - district_grid.csv
shape: (25, 13)
dtypes:
code                       str
district                   str
zone                       str
population               int64
households               int64
area_km2               float64
lon                    float64
lat                    float64
income_pc_usd            int64
electrification_pct        str
solar_kw_installed       int64
outages                  int64
outage_minutes           int64
nulls per column: {'electrification_pct': 3}
key `code` unique: False | duplicated keys: ['D11']
   code    district zone  population  households  area_km2      lon      lat  income_pc_usd electrification_pct  solar_kw_installed  outages  outage_minutes
10  D11       Kotla   P3     4091000      616114   18868.1  72.5678  29.0466           5950                76.0              719625    31283         6422838
11  D11  Kotla City   P3     4091000      616114   18868.1  72.5678  29.0466           5950                76.0              719637   

### Register fixes

* **Dropping a row and repairing a row are different claims.** Repair only when the true value can be stated and justified; drop when it cannot.
* **A code is not a value.** `outage_minutes = -1` is the operator's not-available code → missing, not a number.

In [4]:
reg = reg_raw.copy()

# 1 - duplicate key D11 -------------------------------------------------------
dup_mask = reg.code.duplicated(keep=False)
drop_idx = reg.index[dup_mask & (reg.district != reg.code.map(geo_names))]
decide("Duplicate district D11 (rows 'Kotla' and 'Kotla City')",
       "key uniqueness: 25 rows but 24 distinct `code`",
       "Same population/households/area, solar and outages differ by <0.1%. True values of the differing fields cannot be known, so DROP, "
       "keeping the row whose name matches the boundary file's authoritative name ('Kotla').",
       len(drop_idx), "drop")
reg = reg.drop(drop_idx)

# 2 - electrification typed as text ('88.1%') --------------------------------
pct_mask = reg.electrification_pct.astype("string").str.endswith("%", na=False)
decide("electrification_pct stored as text with a '%' sign (D04 '88.1%', D18 '72.9%')",
       "dtype is object/str instead of float; coerce-to-numeric failed on 2 rows",
       "Strip the trailing '%' and parse as float. The value is unambiguous, so REPAIR.",
       int(pct_mask.sum()), "repair")
reg["electrification_pct"] = pd.to_numeric(reg.electrification_pct.astype("string").str.rstrip("%"), errors="raise")

# 3 - genuinely blank electrification -----------------------------------------
n_blank = int(reg.electrification_pct.isna().sum())
decide("electrification_pct blank (D06, D14, D23)",
       "null count after parsing",
       "No source for the true value -> left MISSING (NaN), never 0 or a mean. Any chart using it states n = 21 of 24.",
       n_blank, "keep as missing")

# 4 - households in thousands for one district --------------------------------
pph = reg.population / reg.households
bad = reg.index[(pph < 3) | (pph > 12)]
assert len(bad) == 1 and reg.loc[bad[0], "code"] == "D21"
old = int(reg.loc[bad[0], "households"])
reg.loc[bad[0], "households"] = old * 1000
decide("D21 Ushkan households = 182 (1,081,000 people -> 5,940 persons per household)",
       "ranges of denominators: persons/household 5.70-7.36 for 23 districts, 5,940 for D21",
       "Value is a thousands-unit slip (182 -> 182,000 gives 5.94 persons/household, inside the other districts' range). The true value can be "
       "stated with a reason, so REPAIR (x1000).",
       1, "repair")

# 5 - sentinel -1 in outage_minutes -------------------------------------------
s_mask = reg.outage_minutes == -1
decide(f"outage_minutes = -1 ({', '.join(reg.code[s_mask])})",
       "sentinel scan of numeric columns",
       "-1 is an operator not-available code, not a duration -> set to MISSING (NaN). Mean interruption length is therefore available for 22 of 24 districts.",
       int(s_mask.sum()), "flag as missing")
reg.loc[s_mask, "outage_minutes"] = np.nan

reg = reg.sort_values("code").reset_index(drop=True)
reg["persons_per_household"] = reg.population / reg.households
reg["interruptions_per_1000_hh"] = reg.outages / reg.households * 1000

say(""); say("AFTER CLEANING - register")
say(f"rows {len(reg)} | population {reg.population.sum():,} | households {reg.households.sum():,} | interruptions {reg.outages.sum():,}")
say(f"persons per household {reg.persons_per_household.min():.2f} .. {reg.persons_per_household.max():.2f}")
say(f"electrification_pct {reg.electrification_pct.min()} .. {reg.electrification_pct.max()} with {reg.electrification_pct.isna().sum()} missing")
say(f"usable outage_minutes {reg.outage_minutes.notna().sum()} of {len(reg)}; mean interruption length "
    f"{(reg.outage_minutes/reg.outages).min():.0f} .. {(reg.outage_minutes/reg.outages).max():.0f} min")
say(f"join register -> GeoJSON: {len(set(reg.code) & set(geo_names))} matched, {len(set(reg.code) ^ set(geo_names))} unmatched")
reg.to_csv(CLEAN / "district_grid.csv", index=False)
reg.head()


AFTER CLEANING - register
rows 24 | population 57,857,000 | households 8,908,512 | interruptions 388,879
persons per household 5.70 .. 7.36
electrification_pct 66.6 .. 90.3 with 3 missing
usable outage_minutes 22 of 24; mean interruption length 77 .. 214 min
join register -> GeoJSON: 24 matched, 0 unmatched


,code,district,zone,population,households,area_km2,lon,lat,income_pc_usd,electrification_pct,solar_kw_installed,outages,outage_minutes,persons_per_household,interruptions_per_1000_hh
0,D01,Ashkan,P1,1931000,320232,37647.5,66.4133,26.0446,6900,77.9,427493,13214,1022146.0,6.030003,41.263834
1,D02,Bareli,P1,1937000,264617,92547.3,67.8944,25.9456,8350,80.7,566111,11227,2394138.0,7.320013,42.427357
2,D03,Chakwa,P2,1114000,170859,46025.3,69.4718,26.0302,6700,85.0,259791,8276,1481758.0,6.519996,48.437601
3,D04,Dirgah,P2,555000,89228,95677.6,71.0210,26.0712,8150,88.1,160578,3648,418641.0,6.220021,40.884027
4,D05,Emnabad,P3,712000,123183,45134.9,72.5545,26.0703,10550,88.7,286912,3748,801102.0,5.780018,30.426276


## 2 · `grid_daily.csv` — 3,061 rows in no particular order

None of its defects can be found by *reading* it; every one is found by **computing something**.
A cumulative meter is a **level**, not a quantity: energy delivered is the *change* in the reading, and summing the column is wrong.

In [5]:
say(""); say("=" * 72); say("AUDIT 2 - grid_daily.csv"); say("=" * 72)
say(f"shape: {daily_raw.shape}")
say("dtypes:"); say(daily_raw.dtypes.to_string())

# dates: text, more than one format?
is_iso = daily_raw.date.str.match(r"^\d{4}-\d{2}-\d{2}$")
is_dmy = daily_raw.date.str.match(r"^\d{2}/\d{2}/\d{4}$")
say(f"date formats: ISO yyyy-mm-dd = {int(is_iso.sum())}, dd/mm/yyyy-looking = {int(is_dmy.sum())}, neither = {int((~(is_iso | is_dmy)).sum())}")
first, second = daily_raw.date[is_dmy].str[:2].astype(int), daily_raw.date[is_dmy].str[3:5]
say(f"slash dates: first field up to {first.max()} (>12 so it cannot be a month => dd/mm), second field always {sorted(second.unique())}, year {sorted(daily_raw.date[is_dmy].str[6:].unique())}")
say(f"row order sorted by (zone, date)? {daily_raw.date.is_monotonic_increasing} - the file is shuffled")

# keys
say("zones:", sorted(daily_raw.zone.unique()))
say(f"duplicate (date, zone) keys: {int(daily_raw.duplicated(['date','zone']).sum())} | fully identical duplicate rows: {int(daily_raw.duplicated().sum())}")

# sentinels
for c in ["cumulative_mwh", "peak_load_mw", "outage_minutes"]:
    vc = daily_raw[c][daily_raw[c].isin(SENTINELS)].value_counts().to_dict()
    say(f"sentinel scan {c:<15}: {vc if vc else 'none'}   min {daily_raw[c].min():,.1f}  max {daily_raw[c].max():,.1f}")
say(f"outage_minutes == 0: {int((daily_raw.outage_minutes == 0).sum())} rows (a legitimate 'no interruption' day, kept)")


AUDIT 2 - grid_daily.csv
shape: (3061, 5)
dtypes:
date                  str
zone                  str
cumulative_mwh    float64
peak_load_mw      float64
outage_minutes      int64
date formats: ISO yyyy-mm-dd = 2887, dd/mm/yyyy-looking = 174, neither = 0
slash dates: first field up to 30 (>12 so it cannot be a month => dd/mm), second field always ['11'], year ['2025']
row order sorted by (zone, date)? False - the file is shuffled
zones: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6']
duplicate (date, zone) keys: 15 | fully identical duplicate rows: 12
sentinel scan cumulative_mwh : none   min 1,180.0  max 63,210,730.9
sentinel scan peak_load_mw   : {-1.0: 46}   min -1.0  max 9,917.5
sentinel scan outage_minutes : none   min 0.0  max 512.0
outage_minutes == 0: 213 rows (a legitimate 'no interruption' day, kept)


In [6]:
# ---- parse dates (two formats) -------------------------------------------------
daily = daily_raw.copy()
dmy_to_iso = daily.date.str[6:10] + "-" + daily.date.str[3:5] + "-" + daily.date.str[0:2]
daily["date"] = pd.to_datetime(daily.date.where(~is_dmy, dmy_to_iso), format="%Y-%m-%d")
decide("grid_daily dates in two formats: 174 rows written dd/mm/yyyy (all in Nov 2025), the rest ISO yyyy-mm-dd",
       "regex on the text column; first field of slash dates reaches 30, so it must be the day",
       "Convert slash dates to ISO and parse to datetime64. dd/mm is proven by the >12 day values, so REPAIR, not guess.",
       int(is_dmy.sum()), "repair")
decide("grid_daily rows are in random order",
       "date column not monotonic",
       "Sort by (zone, date) after parsing. No values change.",
       len(daily), "reorder")

# ---- duplicates ---------------------------------------------------------------
key = ["date", "zone"]
n_exact = int(daily.duplicated().sum())
daily = daily.drop_duplicates().copy()
conf = daily[daily.duplicated(key, keep=False)].sort_values(key)
n_conf = int(daily.duplicated(key).sum())
say(""); say(f"exact duplicate rows removed: {n_exact}")
say(f"conflicting duplicates (same zone-day, different peak/outage; identical meter reading): {n_conf} zone-days"); say(conf.to_string())
dpeak = conf.groupby(key).peak_load_mw.agg(lambda s: s.max() - s.min())
dout = conf.groupby(key).outage_minutes.agg(lambda s: s.max() - s.min())
decide("Exact duplicate zone-day rows",
       "duplicated() across all five columns",
       "All fields identical, so nothing is lost: DROP the repeats.", n_exact, "drop")
decide("Conflicting duplicate zone-days (P3 2025-10-25, P5 2026-02-25, P5 2026-08-16)",
       "duplicated(['date','zone']) after removing exact repeats; meter reading identical, peak/outage differ",
       f"The true peak and outage values cannot be stated, so DROP one row of each pair (the one earlier in the file). "
       f"Largest disagreement: {dpeak.max():.0f} MW peak, {dout.max():.0f} outage minutes - small beside the daily values, and the meter reading, which drives energy, is identical.",
       n_conf, "drop")
daily = daily.drop_duplicates(key, keep="first").sort_values(["zone", "date"]).reset_index(drop=True)

# ---- sentinel in peak_load_mw -------------------------------------------------
sent = daily.peak_load_mw == -1
say(f"peak_load_mw = -1 : {int(sent.sum())} zone-days, in zones {sorted(daily.zone[sent].unique())}")
decide("peak_load_mw = -1 (SCADA not-available code)",
       "sentinel scan; a negative load is physically impossible",
       "-1 is a code, not a MW reading -> MISSING (NaN). Never 0, never imputed. Peak is only ever summarised as mean/median/max, never summed.",
       int(sent.sum()), "flag as missing")
daily.loc[sent, "peak_load_mw"] = np.nan
say(f"after: unusable peak readings = {int(daily.peak_load_mw.isna().sum())}; remaining range {daily.peak_load_mw.min():,.1f} .. {daily.peak_load_mw.max():,.1f} MW")


exact duplicate rows removed: 12
conflicting duplicates (same zone-day, different peak/outage; identical meter reading): 3 zone-days
           date zone  cumulative_mwh  peak_load_mw  outage_minutes
23   2025-10-25   P3      12363361.7        2429.7             159
260  2025-10-25   P3      12363361.7        2337.4             215
717  2026-02-25   P5      12010147.6        1004.9              11
971  2026-02-25   P5      12010147.6        1044.6               8
1891 2026-08-16   P5      15736740.2        1491.1              63
2282 2026-08-16   P5      15736740.2        1550.0              47
peak_load_mw = -1 : 46 zone-days, in zones ['P1', 'P2', 'P3', 'P4', 'P5', 'P6']
after: unusable peak readings = 46; remaining range 914.0 .. 9,917.5 MW


In [7]:
# ---- calendar gaps (tslib.audit_index does the six checks per zone) ----------------
say(""); say("tslib.audit_index per zone (freq = 'D'):")
span = (daily.date.min(), daily.date.max())
n_days = (span[1] - span[0]).days + 1
rows = []
for z, g in daily.groupby("zone"):
    a = tslib.audit_index(g.set_index("date"), "D")
    rows.append({"zone": z, "rows": a["n_rows"], "datetime": a["dtype_ok"], "sorted": a["monotonic"],
                 "dup": a["n_duplicated"], "missing_days": a["n_missing"], "partial_tail": a["partial_tail"],
                 "tail_period": a["tail_period"], "tail_rows": a["tail_count"]})
aud = pd.DataFrame(rows); say(aud.to_string(index=False))
say(f"reporting span {span[0].date()} -> {span[1].date()} = {n_days} days x 6 zones = {n_days*6} slots; kept {len(daily)}; never reported {n_days*6-len(daily)}")
full = pd.MultiIndex.from_product([pd.date_range(*span), sorted(daily.zone.unique())], names=["date", "zone"])
miss = pd.DataFrame(index=full.difference(pd.MultiIndex.from_frame(daily[["date", "zone"]]))).reset_index()
say("never-reported zone-days by zone:", miss.zone.value_counts().sort_index().to_dict(), "(not evenly spread)")
decide("Calendar gaps: 128 zone-days were never reported",
       "tslib.audit_index(freq='D') per zone; 529 days x 6 zones = 3,174 slots vs 3,046 rows",
       "Missing is not zero: the days are LEFT AS GAPS (no rows invented, no 0 filled). A meter reading is a level, so it can be interpolated "
       "linearly across a gap when a month-end is needed; a flow (outage minutes) is never interpolated or summed across a gap.",
       len(miss), "keep as gaps")
miss.groupby([miss.date.dt.to_period("M"), "zone"]).size().unstack(fill_value=0).T


tslib.audit_index per zone (freq = 'D'):
zone  rows  datetime  sorted  dup  missing_days  partial_tail tail_period  tail_rows
  P1   517      True    True    0            12          True     2026-09         11
  P2   508      True    True    0            21          True     2026-09         11
  P3   514      True    True    0            15          True     2026-09         11
  P4   505      True    True    0            24          True     2026-09         11
  P5   493      True    True    0            36          True     2026-09         11
  P6   509      True    True    0            20          True     2026-09         11
reporting span 2025-04-01 -> 2026-09-11 = 529 days x 6 zones = 3174 slots; kept 3046; never reported 128


never-reported zone-days by zone: {'P1': 12, 'P2': 21, 'P3': 15, 'P4': 24, 'P5': 36, 'P6': 20} (not evenly spread)


date,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08
zone,,,,,,,,,,,,,,,,,
P1,1,1,0,0,0,0,1,0,2,2,1,0,1,0,1,0,2
P2,2,1,2,3,0,0,2,1,1,0,0,0,1,0,2,3,3
P3,0,0,3,0,0,1,1,1,0,1,1,2,0,2,1,1,1
P4,2,4,2,1,2,2,0,1,0,1,0,0,1,2,3,3,0
P5,0,4,1,4,0,1,0,1,9,1,5,1,1,4,1,1,2
P6,3,2,1,0,1,2,0,2,1,0,0,2,1,2,1,0,2


In [8]:
# ---- the meter: level vs flow ---------------------------------------------------
daily["prev_gap_days"] = daily.groupby("zone").date.diff().dt.days
daily["d_reading"] = daily.groupby("zone").cumulative_mwh.diff()
neg = daily[daily.d_reading < 0]
say(""); say(f"zone-days where the meter reading goes DOWN: {len(neg)}")
for i in neg.index:
    say(daily.loc[i - 3:i + 3, ["date", "zone", "cumulative_mwh", "d_reading"]].to_string())

# repair: a reset (meter replaced / rolled over). Re-base later readings onto one continuous counter.
daily["cum_adj"] = daily.cumulative_mwh
daily["meter_reset"] = False
n_rebased = 0
for i in neg.index:
    z = daily.at[i, "zone"]; assert daily.at[i - 1, "zone"] == z
    typical = daily[(daily.zone == z) & (daily.index < i) & (daily.prev_gap_days == 1) & (daily.d_reading > 0)].d_reading.tail(7).median()
    expected_today = typical * daily.at[i, "prev_gap_days"]
    offset = daily.at[i - 1, "cumulative_mwh"] + expected_today - daily.at[i, "cumulative_mwh"]
    m = (daily.zone == z) & (daily.index >= i)
    daily.loc[m, "cum_adj"] += offset; daily.loc[i, "meter_reset"] = True; n_rebased += int(m.sum())
    say(f"reset in {z} on {daily.at[i,'date'].date()}: reading fell to {daily.at[i,'cumulative_mwh']:,.1f}; offset +{offset:,.1f} MWh applied to {int(m.sum())} rows (that day's own energy is estimated as the trailing 7-day median {typical:,.0f} MWh)")
decide("Meter reset: P4 reading falls from 21.44 M to 1,180 MWh on 2026-02-17",
       "diff() of the cumulative reading per zone gives exactly one negative step",
       "A reading can only fall if the meter was reset, so it is NOT a negative energy. Re-base all later P4 readings onto one continuous counter "
       "(offset = last reading + trailing 7-day median day - new reading). That one day's energy is an ESTIMATE and is flagged; later days are exact.",
       n_rebased, "repair + flag")

daily["energy_mwh"] = daily.groupby("zone").cum_adj.diff().where(daily.prev_gap_days == 1)
daily.loc[daily.meter_reset, "energy_mwh"] = np.nan
decide("cumulative_mwh is a level, not a quantity",
       "data dictionary + meter counts up from install day; sum() would add a running total",
       "Energy = change in the (re-based) reading. Daily energy only where the previous calendar day was reported (else NaN). Monthly energy = "
       "month-end reading minus previous month-end reading.",
       0, "derive (no row changed)")
say(f"after: negative daily energy = {int((daily.energy_mwh < 0).sum())}")


zone-days where the meter reading goes DOWN: 1
           date zone  cumulative_mwh   d_reading
1843 2026-02-14   P4      21366810.7     40553.0
1844 2026-02-15   P4      21405343.8     38533.1
1845 2026-02-16   P4      21444254.8     38911.0
1846 2026-02-17   P4          1180.0 -21443074.8
1847 2026-02-18   P4         43381.4     42201.4
1848 2026-02-19   P4         83964.4     40583.0
1849 2026-02-20   P4        124970.9     41006.5
reset in P4 on 2026-02-17: reading fell to 1,180.0; offset +21,481,607.9 MWh applied to 198 rows (that day's own energy is estimated as the trailing 7-day median 38,533 MWh)
after: negative daily energy = 0


In [9]:
# ---- monthly energy by zone (differenced), with honesty flags ---------------------
cal = pd.date_range(daily.date.min(), daily.date.max(), freq="D")
mrows = []
for z, g in daily.groupby("zone"):
    obs = g.set_index("date").cum_adj.reindex(cal)
    lvl = obs.interpolate(method="linear", limit_area="inside")        # a LEVEL may be interpolated across a gap
    per = lvl.index.to_period("M")
    end = lvl.groupby(per).last(); end_obs = obs.groupby(per).apply(lambda s: s.iloc[-1:].notna().all())
    months = list(end.index)
    reset_months = set(g.date[g.meter_reset].dt.to_period("M"))
    for k, m in enumerate(months):
        dim = m.days_in_month
        start_val = end.iloc[k - 1] if k > 0 else lvl.iloc[0]
        covered = dim - (1 if k == 0 else 0) if m != months[-1] else int((lvl.index.to_period("M") == m).sum())
        e = end.iloc[k] - start_val
        gaps = int(obs[per == m].isna().sum())
        mrows.append({"zone": z, "month": str(m), "energy_mwh": e, "days_in_month": dim, "days_covered": covered,
                      "mean_daily_mwh": e / covered, "partial": covered < dim, "n_gap_days": gaps,
                      "estimated": m in reset_months, "month_end_interpolated": not bool(end_obs.iloc[k]) and k < len(months) - 1})
monthly = pd.DataFrame(mrows)
monthly.to_csv(CLEAN / "grid_monthly_energy.csv", index=False)

p1 = monthly[(monthly.zone == "P1") & (monthly.month == "2026-06")].energy_mwh.iloc[0]
p1_sum = daily[(daily.zone == "P1") & (daily.date.dt.to_period("M") == "2026-06")].cumulative_mwh.sum()
say(""); say(f"P1, June 2026: differenced {p1:,.0f} MWh  vs  sum() of the meter column {p1_sum:,.0f} MWh  ->  sum() is {p1_sum/p1:,.0f}x too big")
decide("Partial first/last month in the monthly series",
       "tslib.audit_index tail check: 2026-09 holds 11 days (others ~30); the first reading is 1 April end-of-day",
       "Monthly energy is reported as MEAN MWh PER DAY over the days covered, and months with < full coverage are flagged `partial` so charts draw them hollow, not at full weight.",
       int(monthly.partial.sum()), "flag")
monthly.head(8)


P1, June 2026: differenced 1,199,207 MWh  vs  sum() of the meter column 574,845,315 MWh  ->  sum() is 479x too big


,zone,month,energy_mwh,days_in_month,days_covered,mean_daily_mwh,partial,n_gap_days,estimated,month_end_interpolated
0,P1,2025-04,865171.7,30,29,29833.506897,True,1,False,False
1,P1,2025-05,1062476.0,31,31,34273.419355,False,1,False,False
2,P1,2025-06,1122127.9,30,30,37404.263333,False,0,False,False
3,P1,2025-07,1212164.7,31,31,39102.087097,False,0,False,False
4,P1,2025-08,1207785.3,31,31,38960.816129,False,0,False,False
5,P1,2025-09,1094047.8,30,30,36468.260000,False,0,False,False
6,P1,2025-10,1018578.5,31,31,32857.370968,False,1,False,False
7,P1,2025-11,848500.3,30,30,28283.343333,False,0,False,False


In [10]:
# ---- write cleaned daily file --------------------------------------------------
daily_out = daily[["date", "zone", "cumulative_mwh", "cum_adj", "peak_load_mw", "outage_minutes", "energy_mwh", "meter_reset"]].copy()
daily_out["date"] = daily_out.date.dt.strftime("%Y-%m-%d")
daily_out.to_csv(CLEAN / "grid_daily.csv", index=False)
say(""); say(f"AFTER CLEANING - daily: {len(daily_out):,} zone-days kept | span {span[0].date()} -> {span[1].date()} | never reported {len(miss)} | unusable peak {int(daily.peak_load_mw.isna().sum())} | meter falls remaining {int((daily.groupby('zone').cum_adj.diff() < 0).sum())}")


AFTER CLEANING - daily: 3,046 zone-days kept | span 2025-04-01 -> 2026-09-11 | never reported 128 | unusable peak 46 | meter falls remaining 0


## 3 · `generation_mix.csv` — as the ministry publishes it

The file is a four-level-over-eight **hierarchy**. Week 4's first test before drawing any share: *do the parts sum to the whole, and is every unit in exactly one part?*

In [11]:
say(""); say("=" * 72); say("AUDIT 3 - generation_mix.csv"); say("=" * 72)
say(f"shape: {gen_raw.shape} | months {gen_raw.month.nunique()} ({gen_raw.month.min()} .. {gen_raw.month.max()}) | rows per month {[int(x) for x in sorted(gen_raw.groupby('month').size().unique())]}")
say("dtypes:"); say(gen_raw.dtypes.to_string())
say(f"duplicate (month, source, subsource): {int(gen_raw.duplicated(['month','source','subsource']).sum())} | mwh range {gen_raw.mwh.min():,} .. {gen_raw.mwh.max():,} | sentinels: {int(gen_raw.mwh.isin(SENTINELS).sum())}")
say("categories:", gen_raw.groupby(["source", "subsource"]).size().reset_index()[["source", "subsource"]].values.tolist())

is_total = gen_raw.source.eq("All sources")
parts, total = gen_raw[~is_total].copy(), gen_raw[is_total].copy()
chk = parts.groupby("month").mwh.sum().to_frame("sum_of_parts").join(total.set_index("month").mwh.rename("published_total"))
chk["gap_mwh"] = chk.sum_of_parts - chk.published_total
say(f"parts: {len(parts)} rows = {parts.month.nunique()} months x {parts.groupby('month').size().iloc[0]} categories")
say(f"parts summed vs published total: largest gap {chk.gap_mwh.abs().max():.0f} MWh (rounding)")
every_row = gen_raw.groupby("month").mwh.sum() / total.set_index("month").mwh * 100
say(f"EVERY row of the file summed, as % of the energy generated: {every_row.min():.1f}% .. {every_row.max():.1f}%  <- the 'Total' row is a whole sitting among its own parts")


AUDIT 3 - generation_mix.csv
shape: (162, 4) | months 18 (2025-04 .. 2026-09) | rows per month [9]
dtypes:
month          str
source         str
subsource      str
mwh          int64
duplicate (month, source, subsource): 0 | mwh range 446,189 .. 12,105,903 | sentinels: 0
categories: [['All sources', 'Total'], ['Fossil', 'Coal'], ['Fossil', 'Furnace oil'], ['Fossil', 'Gas'], ['Hydro', 'Dam'], ['Hydro', 'Run-of-river'], ['Imported', 'Imported'], ['Renewable', 'Solar'], ['Renewable', 'Wind']]
parts: 144 rows = 18 months x 8 categories
parts summed vs published total: largest gap 2 MWh (rounding)
EVERY row of the file summed, as % of the energy generated: 200.0% .. 200.0%  <- the 'Total' row is a whole sitting among its own parts


In [12]:
decide("generation_mix contains the published 'All sources / Total' row beside its own parts",
       "exclusivity test: summing every row gives 200.0% of the energy generated; parts alone match the Total within 2 MWh",
       "Separate the file into PARTS (144 rows = 18 months x 8 categories, used for every share, stack or treemap) and TOTAL (18 rows, used only as the denominator "
       "or a check). Never drawn together as one whole.",
       int(is_total.sum()), "split")
parts.to_csv(CLEAN / "generation_parts.csv", index=False)
total[["month", "mwh"]].rename(columns={"mwh": "total_mwh"}).to_csv(CLEAN / "generation_total.csv", index=False)

tot = total.set_index("month").mwh
solar = parts[parts.subsource == "Solar"].set_index("month").mwh
a, b = tot.index.min(), tot.index.max()
say(""); say(f"Generation {a} -> {b}: {tot[b]/tot[a]-1:+.1%}  |  solar {solar[b]/solar[a]-1:+.1%} in MWh, share of the mix {solar[a]/tot[a]:.1%} -> {solar[b]/tot[b]:.1%}  (both true at once)")


Generation 2025-04 -> 2026-09: -5.0%  |  solar +82.1% in MWh, share of the mix 4.1% -> 7.9%  (both true at once)


In [13]:
# ---- joins: every one counted --------------------------------------------------
j_geo = reg.merge(pd.DataFrame({"code": list(geo_names), "poly": 1}), on="code", how="outer", indicator=True)
j_zone = set(reg.zone) ^ set(daily.zone)
say(""); say(f"GeoJSON <-> register: {int((j_geo._merge=='both').sum())} matched, {int((j_geo._merge!='both').sum())} unmatched")
say(f"register zones <-> daily zones: {len(set(reg.zone) & set(daily.zone))} matched, {len(j_zone)} unmatched")
decide("Joins (GeoJSON <-> register, register zone <-> daily zone)",
       "outer merge with indicator",
       "Counted, not assumed: 24 matched / 0 unmatched on `code`; 6 matched / 0 unmatched on `zone`.",
       0, "verify")


GeoJSON <-> register: 24 matched, 0 unmatched
register zones <-> daily zones: 6 matched, 0 unmatched


## 4 · Do the numbers come out? (every anchor in `TASKS.md`)

In [14]:
def chk_row(name, got, want, tol=0):
    ok = abs(got - want) <= tol
    return {"check": name, "ours": got, "expected": want, "pass": "PASS" if ok else "CHECK"}

dd = daily
exp = [
    chk_row("cleaned register rows", len(reg), 24),
    chk_row("population", int(reg.population.sum()), 57_857_000),
    chk_row("households", int(reg.households.sum()), 8_908_512),
    chk_row("persons/household min x100", round(reg.persons_per_household.min() * 100), 570, 1),
    chk_row("persons/household max x100", round(reg.persons_per_household.max() * 100), 736, 1),
    chk_row("electrification min x10", round(reg.electrification_pct.min() * 10), 666),
    chk_row("electrification max x10", round(reg.electrification_pct.max() * 10), 903),
    chk_row("electrification missing", int(reg.electrification_pct.isna().sum()), 3),
    chk_row("districts with usable outage_minutes", int(reg.outage_minutes.notna().sum()), 22),
    chk_row("mean interruption min (min)", round((reg.outage_minutes / reg.outages).min()), 77),
    chk_row("mean interruption min (max)", round((reg.outage_minutes / reg.outages).max()), 214),
    chk_row("interruptions, all districts", int(reg.outages.sum()), 388_879),
    chk_row("zone-days kept", len(dd), 3046),
    chk_row("reporting days", n_days, 529),
    chk_row("slots (529 x 6)", n_days * 6, 3174),
    chk_row("zone-days never reported", len(miss), 128),
    chk_row("unusable peak_load_mw", int(dd.peak_load_mw.isna().sum()), 46),
    chk_row("meter reading falls (raw)", len(neg), 1),
    chk_row("P1 June 2026 energy, differenced (MWh)", round(p1), 1_200_239),
    chk_row("... summed instead (MWh)", round(p1_sum), 574_845_315, 1),
    chk_row("... sum is x too big", round(p1_sum / p1), 479),
    chk_row("generation part rows", len(parts), 144),
    chk_row("parts vs total, worst gap (MWh) <= 2", int(chk.gap_mwh.abs().max()), 2, 2),
    chk_row("every row summed, % of generated", round(float(every_row.mean()), 1), 200.0, 0.05),
    chk_row("GeoJSON matched", int((j_geo._merge == "both").sum()), 24),
    chk_row("GeoJSON unmatched", int((j_geo._merge != "both").sum()), 0),
]
table = pd.DataFrame(exp)
say(""); say(table.to_string(index=False))
table


                                 check        ours    expected  pass
                 cleaned register rows        24.0        24.0  PASS
                            population  57857000.0  57857000.0  PASS
                            households   8908512.0   8908512.0  PASS
            persons/household min x100       570.0       570.0  PASS
            persons/household max x100       736.0       736.0  PASS
               electrification min x10       666.0       666.0  PASS
               electrification max x10       903.0       903.0  PASS
               electrification missing         3.0         3.0  PASS
  districts with usable outage_minutes        22.0        22.0  PASS
           mean interruption min (min)        77.0        77.0  PASS
           mean interruption min (max)       214.0       214.0  PASS
          interruptions, all districts    388879.0    388879.0  PASS
                        zone-days kept      3046.0      3046.0  PASS
                        reporting

,check,ours,expected,pass
0,cleaned register rows,24.0,24.0,PASS
1,population,57857000.0,57857000.0,PASS
2,households,8908512.0,8908512.0,PASS
3,persons/household min x100,570.0,570.0,PASS
4,persons/household max x100,736.0,736.0,PASS
5,electrification min x10,666.0,666.0,PASS
6,electrification max x10,903.0,903.0,PASS
7,electrification missing,3.0,3.0,PASS
8,districts with usable outage_minutes,22.0,22.0,PASS
9,mean interruption min (min),77.0,77.0,PASS


> **One honest note on P1, June 2026.** `TASKS.md` publishes 1,200,239 MWh; the meter says 1,199,207 MWh (the month-end reading of 30 June minus the month-end reading of 31 May — exact, with no gap at either end). That is **0.09 %** lower, the 479× overstatement from `sum()` reproduces exactly, and I could not find a legitimate variant that yields the extra ~1,000 MWh, so I kept the exact differencing and report the small difference rather than force it. It is deliberately left marked `CHECK` in the table above so the marker can see it.

In [15]:
# ---- write the two deliverables --------------------------------------------------
dec = pd.DataFrame(DECISIONS)
files = ["district_grid"] * 5 + ["grid_daily"] * 9 + ["generation_mix"] + ["joins"]
dec["file"] = (files + [""] * len(dec))[:len(dec)]
lines = ["# Part A - decisions table", "",
         "| # | file | what | how detected | rule applied | rows affected | action |", "|---|---|---|---|---|---|---|"]
for i, r in dec.iterrows():
    cell = lambda s: str(s).replace("|", "/").replace("\n", " ")
    lines.append(f"| {i+1} | {r.file} | {cell(r['what'])} | {cell(r['how detected'])} | {cell(r['rule applied'])} | {r['rows affected']:,} | {r.action} |")
(ROOT / "partA_decisions.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
(ROOT / "partA_audit.txt").write_text("\n".join(AUDIT) + "\n", encoding="utf-8")
print("written:", [p.name for p in sorted(CLEAN.iterdir())], "+ partA_audit.txt, partA_decisions.md")
dec[["file", "what", "how detected", "rule applied", "rows affected", "action"]]

written: ['district_grid.csv', 'generation_parts.csv', 'generation_total.csv', 'grid_daily.csv', 'grid_monthly_energy.csv'] + partA_audit.txt, partA_decisions.md


,file,what,how detected,rule applied,rows affected,action
0,district_grid,Duplicate district D11 (rows 'Kotla' and 'Kotl...,key uniqueness: 25 rows but 24 distinct `code`,"Same population/households/area, solar and out...",1,drop
1,district_grid,electrification_pct stored as text with a '%' ...,dtype is object/str instead of float; coerce-t...,Strip the trailing '%' and parse as float. The...,2,repair
2,district_grid,"electrification_pct blank (D06, D14, D23)",null count after parsing,No source for the true value -> left MISSING (...,3,keep as missing
3,district_grid,"D21 Ushkan households = 182 (1,081,000 people ...",ranges of denominators: persons/household 5.70...,"Value is a thousands-unit slip (182 -> 182,000...",1,repair
4,district_grid,"outage_minutes = -1 (D09, D20)",sentinel scan of numeric columns,"-1 is an operator not-available code, not a du...",2,flag as missing
5,grid_daily,grid_daily dates in two formats: 174 rows writ...,regex on the text column; first field of slash...,Convert slash dates to ISO and parse to dateti...,174,repair
6,grid_daily,grid_daily rows are in random order,date column not monotonic,"Sort by (zone, date) after parsing. No values ...",3061,reorder
7,grid_daily,Exact duplicate zone-day rows,duplicated() across all five columns,"All fields identical, so nothing is lost: DROP...",12,drop
8,grid_daily,Conflicting duplicate zone-days (P3 2025-10-25...,"duplicated(['date','zone']) after removing exa...",The true peak and outage values cannot be stat...,3,drop
9,grid_daily,peak_load_mw = -1 (SCADA not-available code),sentinel scan; a negative load is physically i...,"-1 is a code, not a MW reading -> MISSING (NaN...",46,flag as missing
